# Fine-Tuning Embedding Models with Adapters

This notebook demonstrates how to improve retrieval quality by fine-tuning embedding models using lightweight adapter layers — without retraining the full model from scratch.

## What You Will Learn

- How to generate question-answer pairs from a knowledge corpus to create a fine-tuning dataset
- How to fine-tune a local BAAI/bge embedding model using an adapter layer with LlamaIndex
- How to fine-tune an OpenAI embedding model using the same adapter approach
- How to evaluate and compare embedding models before and after fine-tuning using hit rate metrics

## Prerequisites

- Familiarity with RAG pipelines and vector similarity search (covered in earlier lessons)
- Basic understanding of embedding models and how they are used in retrieval
- An OpenAI API key with access to `text-embedding-3-small`

<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/08-Finetune_Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [1]:
!pip install -qU llama-index==0.14.16 openai==2.30.0 google-genai==1.70.0 llama-index-embeddings-openai==0.5.2 \
                chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2 opentelemetry-api==1.38.0\
                llama-index-llms-openai==0.6.26 llama-index-finetuning==0.4.2 llama-index-embeddings-huggingface==0.7.0 \
                llama-index-embeddings-adapter llama-index-llms-azure-openai llama-index-llms-google-genai==0.9.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")

# Download the Dataset


In [3]:
from huggingface_hub import hf_hub_download
file_path = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="ai_tutor_knowledge.jsonl",repo_type="dataset",local_dir="/content")

ai_tutor_knowledge.jsonl: 0.00B [00:00, ?B/s]

In [4]:
import json
# Load the JSONL file
with open(file_path, "r") as file:
    ai_tutor_knowledge = [json.loads(line) for line in file]

len(ai_tutor_knowledge)

762

## LlamaIndex Document

In [5]:
from typing import List
from llama_index.core import Document

def create_docs_from_list(data_list: List[dict]) -> List[Document]:
    documents = []
    for data in data_list:
        documents.append(
            Document(
                doc_id=data["doc_id"],
                text=data["content"],
                metadata={  # type: ignore
                    "url": data["url"],
                    "title": data["name"],
                    "tokens": data["tokens"],
                    "source": data["source"],
                },
                excluded_llm_metadata_keys=[
                    "title",
                    "tokens",
                    "source",
                ],
                excluded_embed_metadata_keys=[
                    "url",
                    "tokens",
                    "source",
                ],
            )
        )
    return documents

doc = create_docs_from_list(ai_tutor_knowledge)

### Splitting Dataset


In [6]:
import random

random.shuffle(doc)
split_index = int(len(doc) * 0.9)

# TRAIN_DOCs and VALIDATION_DOCs
TRAIN_DOCs = doc[:split_index]
VALIDATION_DOCs = doc[split_index:]

# Chunking


In [7]:
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.schema import Document

# Now use the parser
parser = SimpleNodeParser.from_defaults(chunk_size=768, chunk_overlap=64)
TRAIN_NODEs = parser.get_nodes_from_documents(TRAIN_DOCs)
VALIDATION_NODEs = parser.get_nodes_from_documents(VALIDATION_DOCs)

print(len(TRAIN_NODEs), len(VALIDATION_NODEs))

2823 291


In [8]:
# Use a subset of the dataset if testing.

# Test with a few samples; processing the full dataset can be costly depending on the size.
# NOTE: Checkpoints are provided in the lesson, so no need to run the code on the full dataset.

testing =True

if testing:
    TRAIN_NODEs = TRAIN_NODEs[0:200]
    VALIDATION_NODEs = VALIDATION_NODEs[0:40]

# Generate Question


We use a Large Language Model (LLM) to produce questions for each chunk of the dataset. Then we can use these data to train the model to develop embeddings that more accurately represent the types of questions users may ask.


In [9]:
# # Use this block of code if you want to generate the questions for the dataset.
# # Uncomment the following code, and keep in mind to comment all the contents in the next coding block.

# from llama_index.finetuning import generate_qa_embedding_pairs
# from llama_index.llms.openai import OpenAI

# llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})

# # Generate questions for each chunk.

# TRAIN_DATASET = generate_qa_embedding_pairs(TRAIN_NODEs, llm=llm, output_path="./train_dataset.json")

# VALIDATION_DATASET = generate_qa_embedding_pairs(VALIDATION_NODEs, llm=llm, output_path="./val_dataset.json")

In [10]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="jaiganesan/Embedding-model-fine-tuning-dataset", repo_type="dataset",local_dir="/content/")


from llama_index.finetuning import EmbeddingQAFinetuneDataset

# Load the pre-generated questions json files.
TRAIN_DATASET = EmbeddingQAFinetuneDataset.from_json("./train_dataset.json")
VALIDATION_DATASET = EmbeddingQAFinetuneDataset.from_json("./val_dataset.json")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/tmp/ipykernel_19018/10067104.py:5: DeprecationWarning: llama-index-finetuning is deprecated and no longer maintained. It will not receive any further updates.
  from llama_index.finetuning import EmbeddingQAFinetuneDataset


# Load an Embedding Model


In [11]:
from llama_index.core.embeddings import resolve_embed_model

# Load an existing embedding model with a adapter layer on top.
base_embed_model = resolve_embed_model("local:BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
from llama_index.finetuning import EmbeddingAdapterFinetuneEngine
import torch

# Finetune the adapter
finetune_engine = EmbeddingAdapterFinetuneEngine(
    dataset=TRAIN_DATASET,
    embed_model=base_embed_model,
    model_output_path="model_output_test",
    epochs=2,
    verbose=True,
    bias=True,
)

In [ ]:
# Initiate the Finetuning process
finetune_engine.finetune()

> Prepared optimizer, scheduler, and loss model.


Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

Iteration:   0%|          | 0/545 [00:00<?, ?it/s]

> [Epoch 0] Current loss: 1.5561200380325317
> [Epoch 0] Current loss: 1.5454349517822266
> [Epoch 0] Current loss: 1.3019955158233643
> [Epoch 0] Current loss: 1.2101980447769165
> [Epoch 0] Current loss: 1.0392595529556274
> [Epoch 0] Current loss: 1.0869505405426025
> [Epoch 0] Current loss: 1.6084486246109009
> [Epoch 0] Current loss: 1.39484441280365
> [Epoch 0] Current loss: 1.62460196018219
> [Epoch 0] Current loss: 1.8332297801971436
> [Epoch 0] Current loss: 2.3522720336914062
> [Epoch 0] Current loss: 2.365095376968384
> [Epoch 0] Current loss: 2.266875982284546
> [Epoch 0] Current loss: 1.8664630651474
> [Epoch 0] Current loss: 2.3703010082244873
> [Epoch 0] Current loss: 2.3469536304473877
> [Epoch 0] Current loss: 2.4227182865142822
> [Epoch 0] Current loss: 2.3735404014587402
> [Epoch 0] Current loss: 2.2941601276397705
> [Epoch 0] Current loss: 2.3603806495666504
> [Epoch 0] Current loss: 2.3185572624206543
> [Epoch 0] Current loss: 2.3356425762176514
> [Epoch 0] Current

Iteration:   0%|          | 0/545 [00:00<?, ?it/s]

> [Epoch 1] Current loss: 1.5034741163253784
> [Epoch 1] Current loss: 1.4798128604888916
> [Epoch 1] Current loss: 1.293006420135498
> [Epoch 1] Current loss: 1.1630271673202515
> [Epoch 1] Current loss: 1.029118299484253
> [Epoch 1] Current loss: 1.0037192106246948
> [Epoch 1] Current loss: 1.5205399990081787
> [Epoch 1] Current loss: 1.3807647228240967
> [Epoch 1] Current loss: 1.6184275150299072
> [Epoch 1] Current loss: 1.8021920919418335
> [Epoch 1] Current loss: 2.352696418762207
> [Epoch 1] Current loss: 2.3439199924468994
> [Epoch 1] Current loss: 2.264328718185425
> [Epoch 1] Current loss: 1.856845498085022
> [Epoch 1] Current loss: 2.3480913639068604
> [Epoch 1] Current loss: 2.3382058143615723
> [Epoch 1] Current loss: 2.3973591327667236
> [Epoch 1] Current loss: 2.3641161918640137
> [Epoch 1] Current loss: 2.272991418838501
> [Epoch 1] Current loss: 2.337275266647339
> [Epoch 1] Current loss: 2.310944080352783
> [Epoch 1] Current loss: 2.3237786293029785
> [Epoch 1] Curren

## Optional: What Does the Adapter Learn?

In [ ]:
# The adapter is a small trainable linear layer on top of the frozen base embedding model.
# It learns to shift embedding vectors to better separate relevant from irrelevant chunks
# for your specific domain — without retraining the full model.

try:
    import torch
    adapter = finetune_engine.get_finetuned_model()
    # Count trainable vs total parameters
    total_params = sum(p.numel() for p in adapter._model.parameters())
    trainable_params = sum(p.numel() for p in adapter._model.parameters() if p.requires_grad)
    print(f"Adapter total parameters     : {total_params:,}")
    print(f"Adapter trainable parameters : {trainable_params:,}")
    print(f"Base model parameters frozen : {total_params - trainable_params:,}")
    print(f"\nAdapter layer summary:")
    for name, layer in adapter._model.named_modules():
        if name:
            print(f"  {name}: {layer}")
except Exception as e:
    print(f"Could not inspect adapter (run finetune_engine.finetune() first): {e}")

In [ ]:
embed_model = finetune_engine.get_finetuned_model()

# Or, import model from the directory whenever required.
# from llama_index.core.embeddings import LinearAdapterEmbeddingModel
# embed_model = LinearAdapterEmbeddingModel(base_embed_model, "model_output_test")

In [ ]:
print(embed_model)

## Fine tuning OpenAI Embedding Model using Adapter method

In [ ]:
from llama_index.finetuning import EmbeddingAdapterFinetuneEngine
from llama_index.embeddings.openai import OpenAIEmbedding

openai_finetune_engine = EmbeddingAdapterFinetuneEngine(
    dataset=TRAIN_DATASET,
    embed_model=OpenAIEmbedding(model="text-embedding-3-small"),
    model_output_path="model_output_test_openai",
    bias=True,
    epochs=2,
    verbose=True,
)

In [ ]:
openai_finetune_engine.finetune()

openai_embed_model = openai_finetune_engine.get_finetuned_model()

In [ ]:
print(openai_embed_model)

# Evaluate


## Define the Evaluation Functions


In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import TextNode
from llama_index.core import Settings
from tqdm import tqdm
import pandas as pd

def evaluate(dataset, embedding_model, top_k=5, verbose=False):
    corpus = dataset.corpus
    queries = dataset.queries
    relevant_docs = dataset.relevant_docs

    Settings.embed_model = embedding_model

    # Chunking the documents and generating embeddings
    nodes = [TextNode(id_=id_, text=text) for id_, text in corpus.items()]
    index = VectorStoreIndex(nodes, show_progress=True)

    # Define a retriever to answer the questions
    retriever = index.as_retriever(similarity_top_k=top_k)

    eval_results = []

    # Look into each response source to see if the chunk that contains the answer is retrieved.
    for query_id, query in tqdm(queries.items()):
        retrieved_nodes = retriever.retrieve(query)
        retrieved_ids = [node.node.node_id for node in retrieved_nodes]
        expected_id = relevant_docs[query_id][0]

        try:
            rank = retrieved_ids.index(expected_id) + 1
            reciprocal_rank = 1 / rank
        except ValueError:
            rank = None
            reciprocal_rank = 0

        is_hit = expected_id in retrieved_ids

        eval_result = {
            "is_hit": is_hit,
            "retrieved": retrieved_ids,
            "expected": expected_id,
            "query": query_id,
            "rank": rank,
            "reciprocal_rank": reciprocal_rank
        }
        eval_results.append(eval_result)

    return eval_results

## OpenAI Embedding Model Evaluation


In [ ]:
# Load the OpenAI Ada model and evaluate it.
openai_text_embedding_small = OpenAIEmbedding(model="text-embedding-3-small")
openai_embedding_val_results = evaluate(VALIDATION_DATASET, embedding_model=openai_text_embedding_small)

In [ ]:
openai_embedding_val_results = [
    result for result in openai_embedding_val_results if isinstance(result, dict)
]

df_openai = pd.DataFrame(openai_embedding_val_results)

hit_rate_openai = df_openai["is_hit"].mean()
mrr_openai = df_openai["reciprocal_rank"].mean()

print(f"Hit rate: {hit_rate_openai}")
print(f"MRR: {mrr_openai}")

### OpenAI Embedding Model with Fine Tuned Adapter Model Evaluation

In [ ]:
from llama_index.embeddings.adapter import AdapterEmbeddingModel

openai_embed_model = AdapterEmbeddingModel(openai_text_embedding_small, "model_output_test_openai")

val_results_ft_openai = evaluate(VALIDATION_DATASET, embedding_model = openai_embed_model)

In [ ]:
val_results_ft_openai = [
    result for result in val_results_ft_openai if isinstance(result, dict)
]

df_openai_ft = pd.DataFrame(val_results_ft_openai)

hit_rate_openai_ft = df_openai_ft["is_hit"].mean()
mrr_openai_ft = df_openai_ft["reciprocal_rank"].mean()

print(f"Hit rate: {hit_rate_openai_ft}")
print(f"MRR: {mrr_openai_ft}")

## Open Source BAAI Model Evaluation


In [ ]:
# Load the Base model without fine-tuning
base_embed_model = resolve_embed_model("local:BAAI/bge-small-en-v1.5")
bge_val_results = evaluate(VALIDATION_DATASET, embedding_model=base_embed_model)

In [ ]:
bge_val_results = [
    result for result in bge_val_results if isinstance(result, dict)
]

df_bge = pd.DataFrame(bge_val_results)

hit_rate_bge = df_bge["is_hit"].mean()
mrr_bge = df_bge["reciprocal_rank"].mean()

print(f"Hit rate: {hit_rate_bge}")
print(f"MRR: {mrr_bge}")

## FineTuned BAAI Adapter Embedding Model Evaluation


In [ ]:
from llama_index.embeddings.adapter import AdapterEmbeddingModel

# Load the Fine-tuned model.
embed_model = AdapterEmbeddingModel(base_embed_model, "model_output_test")

val_results_finetuned = evaluate(VALIDATION_DATASET, embedding_model=embed_model)

In [ ]:
val_results_finetuned = [
    result for result in val_results_finetuned if isinstance(result, dict)
]

df_finetuned = pd.DataFrame(val_results_finetuned)

hit_rate_finetuned = df_finetuned["is_hit"].mean()
mrr_finetuned = df_finetuned["reciprocal_rank"].mean()

print(f"Hit rate: {hit_rate_finetuned}")
print(f"MRR: {mrr_finetuned}")

## Optional: Fine-Tuning Impact Summary

In [ ]:
# Compare hit rates across all four embedding configurations
import pandas as pd

def compute_hit_rate(results):
    """Compute hit rate from a list of EmbeddingQAFinetuneDataset eval results."""
    hits = [r for r in results if isinstance(r, dict) and r.get("is_hit")]
    total = [r for r in results if isinstance(r, dict)]
    return len(hits) / len(total) if total else 0.0

configs = [
    ("OpenAI base (text-embedding-3-small)", "openai_embedding_val_results"),
    ("OpenAI + adapter (fine-tuned)",         "val_results_ft_openai"),
    ("BAAI/bge base",                          "bge_val_results"),
    ("BAAI/bge + adapter (fine-tuned)",        "val_results_finetuned"),
]

rows = []
for label, var_name in configs:
    results = locals().get(var_name) or globals().get(var_name)
    if results:
        hr = compute_hit_rate(results)
        rows.append({"Model": label, "Hit Rate": f"{hr:.3f}", "Samples": len([r for r in results if isinstance(r, dict)])})
    else:
        rows.append({"Model": label, "Hit Rate": "\u2014", "Samples": "\u2014"})

df = pd.DataFrame(rows)
print("=== Embedding Fine-Tuning \u2014 Hit Rate Comparison ===\n")
print(df.to_string(index=False))
print("\nHigher hit rate = embedding finds the relevant chunk more often.")